In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import matplotlib.colors as colors 

In [ ]:
import numpy as np
import pandas as pd

def fwhm_to_sigma(fwhm):
    return fwhm / (2 * np.sqrt(2 * np.log(2)))


def DoG_from_radius(r, sign, fwhm_c, fwhm_s=None, A_rel=None,  projected_1d=True,):
    """
    Radial Difference-of-Gaussians RF.

    Parameters
    ----------
    r : array-like
        Radial distance from RF center, in degrees.
        For 1D, r can just be x.
        For 2D, r = sqrt(x^2 + y^2).

    sign : float
        +1 for ON-like, -1 for OFF-like.

    fwhm_c : float
        Center FWHM in degrees.

    fwhm_s : float or NaN
        Surround FWHM in degrees. If NaN, no surround is used.

    A_rel : float or NaN
        Relative surround amplitude. If NaN, no surround is used.

    Returns
    -------
    rf : np.ndarray
        RF evaluated at radius r.
    """
    r = np.asarray(r, dtype=float)

    sigma_c = fwhm_to_sigma(fwhm_c)
    rf_c = np.exp(-(r**2) / (2 * sigma_c**2))

    if pd.isna(fwhm_s) or pd.isna(A_rel):
        rf_s = 0.0
    else:
        sigma_s = fwhm_to_sigma(fwhm_s)
        if projected_1d:
            A_eff = A_rel * (sigma_s / sigma_c)
        else:
            A_eff = A_rel
        rf_s = A_eff * np.exp(-(r**2) / (2 * sigma_s**2))

    return sign * (rf_c - rf_s)

def DoG_1D(x, sign, fwhm_c, fwhm_s=None, A_rel=None,projected_1d=True):
    """
    1D cross-section of radial DoG.
    """
    return DoG_from_radius(
        r=x,
        sign=sign,
        fwhm_c=fwhm_c,
        fwhm_s=fwhm_s,
        A_rel=A_rel,
        projected_1d=projected_1d
    )


def DoG_2D(x_grid, y_grid, sign, fwhm_c, fwhm_s=None, A_rel=None, projected_1d=True):
    """
    2D radially symmetric DoG.
    """
    r = np.sqrt(x_grid**2 + y_grid**2)

    return DoG_from_radius(
        r=r,
        sign=sign,
        fwhm_c=fwhm_c,
        fwhm_s=fwhm_s,
        A_rel=A_rel,
        projected_1d=projected_1d
    )